In [1]:
# ============================================================
# MOROCCO vs TURKEY - BILATERAL TRADE ANALYSIS
# ============================================================
# Context: The Minister stated that Turkish engineers 
# are beating Moroccan engineers "5-0" in industrial conquest.
# This analysis examines the bilateral trade data to quantify
# the gap and identify strategic opportunities for Morocco.
# ============================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

pd.options.display.float_format = '{:,.2f}'.format
sns.set_theme(style="whitegrid")

# Create output folder for charts
os.makedirs('charts', exist_ok=True)
os.makedirs('report', exist_ok=True)
os.makedirs('presentation', exist_ok=True)

print("✅ All libraries loaded successfully!")
print("📁 Output directories created: charts/, report/, presentation/")

✅ All libraries loaded successfully!
📁 Output directories created: charts/, report/, presentation/


In [2]:
# ============================================================
# STEP 1: DATA LOADING & CLEANING
# ============================================================

# Load raw CSV files (skip metadata header rows)
df_imp_raw = pd.read_csv('Imports.csv', skiprows=14)
df_exp_raw = pd.read_csv('Exports.csv', skiprows=14)

# Fix unnamed columns
df_imp_raw = df_imp_raw.rename(columns={'Unnamed: 0': 'Code', 'Unnamed: 1': 'Product_Label'})
df_exp_raw = df_exp_raw.rename(columns={'Unnamed: 0': 'Code', 'Unnamed: 1': 'Product_Label'})

# Strip whitespace from column names
df_imp_raw.columns = df_imp_raw.columns.str.strip()
df_exp_raw.columns = df_exp_raw.columns.str.strip()

# Select relevant columns (2021-2024)
years = ['2021', '2022', '2023', '2024']

# --- IMPORTS: Morocco's imports FROM Turkey ---
imp_bilateral_cols = ['Code', 'Product_Label'] + [f'Value in {y}' for y in years]
df_imports = df_imp_raw[imp_bilateral_cols].copy()
for y in years:
    df_imports = df_imports.rename(columns={f'Value in {y}': f'Import_{y}'})

# --- EXPORTS: Morocco's exports TO Turkey ---
exp_bilateral_cols = ['Code'] + [f'Value in {y}' for y in years]
df_exports = df_exp_raw[exp_bilateral_cols].copy()
for y in years:
    df_exports = df_exports.rename(columns={f'Value in {y}': f'Export_{y}'})

# Also get Turkey's world exports and Morocco's world imports/exports
# Turkey's exports to world from Imports.csv
turk_world_cols = ['Code']
for col in df_imp_raw.columns:
    if 'Value in' in col:
        # Get the second set of "Value in" columns (Turkey's exports to world)
        pass

# Merge bilateral data
df = pd.merge(df_imports, df_exports, on='Code', how='outer').fillna(0)

# Convert numeric columns
for y in years:
    df[f'Import_{y}'] = pd.to_numeric(df[f'Import_{y}'], errors='coerce').fillna(0)
    df[f'Export_{y}'] = pd.to_numeric(df[f'Export_{y}'], errors='coerce').fillna(0)

# Remove TOTAL row
df = df[df['Code'] != "'TOTAL"]

# Clean product codes
df['Code_Clean'] = df['Code'].str.replace("'", "").str.strip()

# ============================================================
# SECTOR CLASSIFICATION (HS Code based)
# ============================================================
def classify_sector(code):
    c = str(code).replace("'", "").zfill(2)[:2]
    mapping = {
        '84': 'Machinery & Equipment', '85': 'Electronics & Electrical',
        '87': 'Vehicles & Auto Parts', '90': 'Precision Instruments',
        '86': 'Railway Equipment', '88': 'Aircraft & Spacecraft',
        '72': 'Iron & Steel', '73': 'Iron/Steel Articles',
        '74': 'Copper', '76': 'Aluminum', '75': 'Nickel',
        '39': 'Plastics', '40': 'Rubber',
        '61': 'Knitted Apparel', '62': 'Woven Apparel',
        '63': 'Textile Articles', '64': 'Footwear',
        '54': 'Man-made Filaments', '55': 'Man-made Staple Fibers',
        '52': 'Cotton', '60': 'Knitted Fabrics',
        '27': 'Mineral Fuels & Oils', '25': 'Salt, Sulphur, Cement',
        '26': 'Ores, Slag & Ash',
        '28': 'Inorganic Chemicals', '29': 'Organic Chemicals',
        '31': 'Fertilizers (Phosphates)', '38': 'Chemical Products',
        '30': 'Pharmaceuticals',
        '01': 'Live Animals', '02': 'Meat', '03': 'Fish & Seafood',
        '04': 'Dairy Products', '07': 'Vegetables', '08': 'Fruits & Nuts',
        '10': 'Cereals', '11': 'Milling Products', '12': 'Oil Seeds',
        '17': 'Sugar', '20': 'Vegetable Preparations', '22': 'Beverages',
    }
    return mapping.get(c, 'Other Products')

def classify_macro_sector(code):
    c = str(code).replace("'", "").zfill(2)[:2]
    if c in ['84', '85', '87', '90', '86', '88', '89']:
        return 'High-Tech & Machinery'
    elif c in ['61', '62', '63', '64', '52', '54', '55', '60']:
        return 'Textiles & Apparel'
    elif c in ['01','02','03','04','07','08','09','10','11','12','15','16','17','19','20','21','22','23','24']:
        return 'Agriculture & Food'
    elif c in ['72', '73', '74', '75', '76']:
        return 'Metals & Minerals'
    elif c in ['28', '29', '30', '31', '32', '33', '34', '38']:
        return 'Chemicals & Pharma'
    elif c in ['39', '40']:
        return 'Plastics & Rubber'
    elif c in ['27']:
        return 'Energy & Fuels'
    else:
        return 'Other Industries'

df['Sector'] = df['Code'].apply(classify_sector)
df['Macro_Sector'] = df['Code'].apply(classify_macro_sector)

# Calculate Trade Balance for each year
for y in years:
    df[f'Balance_{y}'] = df[f'Export_{y}'] - df[f'Import_{y}']

# Calculate growth rates
df['Import_Growth_%'] = ((df['Import_2024'] - df['Import_2021']) / (df['Import_2021'] + 1)) * 100
df['Export_Growth_%'] = ((df['Export_2024'] - df['Export_2021']) / (df['Export_2021'] + 1)) * 100

print(f"✅ Data loaded: {len(df)} product categories")
print(f"📊 Years covered: {', '.join(years)}")
print(f"\n{'='*60}")
print(f"TOTAL BILATERAL TRADE SUMMARY (2024)")
print(f"{'='*60}")
total_imports = df['Import_2024'].sum()
total_exports = df['Export_2024'].sum()
total_balance = total_exports - total_imports
print(f"Morocco's Imports from Turkey: ${total_imports:,.0f}K")
print(f"Morocco's Exports to Turkey:   ${total_exports:,.0f}K")
print(f"Trade Balance (Morocco):       ${total_balance:,.0f}K")
print(f"Coverage Ratio:                {(total_exports/total_imports)*100:.1f}%")
print(f"\n⚠️  Morocco has a DEFICIT of ${abs(total_balance):,.0f}K with Turkey")

✅ Data loaded: 96 product categories
📊 Years covered: 2021, 2022, 2023, 2024

TOTAL BILATERAL TRADE SUMMARY (2024)
Morocco's Imports from Turkey: $3,949,791K
Morocco's Exports to Turkey:   $1,172,048K
Trade Balance (Morocco):       $-2,777,743K
Coverage Ratio:                29.7%

⚠️  Morocco has a DEFICIT of $2,777,743K with Turkey


In [3]:
# ============================================================
# STEP 2: OVERALL TRADE EVOLUTION (2021-2024)
# ============================================================

summary = pd.DataFrame({
    'Year': years,
    'Morocco_Imports_from_Turkey': [df[f'Import_{y}'].sum() for y in years],
    'Morocco_Exports_to_Turkey': [df[f'Export_{y}'].sum() for y in years],
})
summary['Trade_Balance'] = summary['Morocco_Exports_to_Turkey'] - summary['Morocco_Imports_from_Turkey']
summary['Deficit_Ratio'] = summary['Morocco_Imports_from_Turkey'] / summary['Morocco_Exports_to_Turkey']
summary['Coverage_%'] = (summary['Morocco_Exports_to_Turkey'] / summary['Morocco_Imports_from_Turkey']) * 100

print("📈 YEAR-BY-YEAR TRADE EVOLUTION")
print("="*70)
print(summary.to_string(index=False))

# --- Chart 1: Trade Evolution (Matplotlib for saving) ---
fig_m, ax1 = plt.subplots(figsize=(10, 5))
x = np.arange(len(years))
w = 0.3
bars1 = ax1.bar(x - w/2, summary['Morocco_Imports_from_Turkey']/1e6, w, label='Imports from Turkey', color='#e74c3c', alpha=0.8)
bars2 = ax1.bar(x + w/2, summary['Morocco_Exports_to_Turkey']/1e6, w, label='Exports to Turkey', color='#2ecc71', alpha=0.8)
ax2 = ax1.twinx()
ax2.plot(x, summary['Trade_Balance']/1e6, 'o--', color='#f39c12', linewidth=2.5, markersize=8, label='Trade Balance')
for i, v in enumerate(summary['Trade_Balance']/1e6):
    ax2.annotate(f'${v:.2f}B', (x[i], v), textcoords="offset points", xytext=(0, 12), ha='center', fontsize=9, fontweight='bold')
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Trade Volume (USD Billion)', fontsize=11)
ax2.set_ylabel('Trade Balance (USD Billion)', fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels(years)
ax1.set_title('Morocco-Turkey Bilateral Trade Evolution (2021-2024)\nSource: ITC / UN COMTRADE | Unit: USD Thousand', fontsize=13, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('charts/01_trade_evolution.png', dpi=200, bbox_inches='tight')
plt.show()

# Also show interactive plotly version
fig1 = make_subplots(specs=[[{"secondary_y": True}]])
fig1.add_trace(go.Bar(x=summary['Year'], y=summary['Morocco_Imports_from_Turkey'], name='Imports from Turkey', marker_color='#e74c3c', opacity=0.8), secondary_y=False)
fig1.add_trace(go.Bar(x=summary['Year'], y=summary['Morocco_Exports_to_Turkey'], name='Exports to Turkey', marker_color='#2ecc71', opacity=0.8), secondary_y=False)
fig1.add_trace(go.Scatter(x=summary['Year'], y=summary['Trade_Balance'], name='Trade Balance', line=dict(color='#f39c12', width=3, dash='dash'), mode='lines+markers+text', text=[f'${v:,.0f}K' for v in summary['Trade_Balance']], textposition='top center'), secondary_y=True)
fig1.update_layout(title='<b>Morocco-Turkey Bilateral Trade Evolution (2021-2024)</b>', barmode='group', template='plotly_white', legend=dict(orientation="h", yanchor="bottom", y=-0.2), height=500, width=900)
fig1.show()
print(f"\n💾 Chart saved: charts/01_trade_evolution.png")

📈 YEAR-BY-YEAR TRADE EVOLUTION
Year  Morocco_Imports_from_Turkey  Morocco_Exports_to_Turkey  Trade_Balance  Deficit_Ratio  Coverage_%
2021                      3381861                     800044       -2581817           4.23       23.66
2022                      3751026                    1101198       -2649828           3.41       29.36
2023                      3610533                    1168384       -2442149           3.09       32.36
2024                      3949791                    1172048       -2777743           3.37       29.67



💾 Chart saved: charts/01_trade_evolution.png


In [ ]:
# ============================================================
# STEP 3: SECTOR-LEVEL ANALYSIS (2024)
# ============================================================

sector_2024 = df.groupby('Macro_Sector').agg({
    'Import_2024': 'sum', 'Export_2024': 'sum', 'Balance_2024': 'sum',
    'Import_2021': 'sum', 'Export_2021': 'sum', 'Balance_2021': 'sum'
}).reset_index()
sector_2024['Coverage_2024_%'] = (sector_2024['Export_2024'] / (sector_2024['Import_2024'] + 1)) * 100
sector_2024['Balance_Change'] = sector_2024['Balance_2024'] - sector_2024['Balance_2021']
sector_2024 = sector_2024.sort_values('Balance_2024')

print("📊 SECTOR ANALYSIS - TRADE BALANCE 2024")
print("="*80)
print(sector_2024[['Macro_Sector', 'Import_2024', 'Export_2024', 'Balance_2024', 'Coverage_2024_%']].to_string(index=False))

# --- Chart 2: Sector Trade Balance (Matplotlib) ---
fig_m2, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in sector_2024['Balance_2024']]
bars = ax.barh(sector_2024['Macro_Sector'], sector_2024['Balance_2024']/1e3, color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(x=0, color='black', linewidth=1.5)
for bar, val in zip(bars, sector_2024['Balance_2024']):
    x_pos = bar.get_width()
    ax.text(x_pos + (50 if x_pos >= 0 else -50), bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}K', va='center', ha='left' if x_pos >= 0 else 'right', fontsize=9)
ax.set_xlabel('Trade Balance (USD Millions)', fontsize=11)
ax.set_title('Trade Balance by Sector: Morocco vs Turkey (2024)\nRed = Deficit | Green = Surplus', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/02_sector_balance_2024.png', dpi=200, bbox_inches='tight')
plt.show()

# Plotly interactive
fig2 = go.Figure()
fig2.add_trace(go.Bar(x=sector_2024['Balance_2024'], y=sector_2024['Macro_Sector'], orientation='h',
    marker_color=colors, text=[f'${v:,.0f}K' for v in sector_2024['Balance_2024']], textposition='outside'))
fig2.update_layout(title='<b>Trade Balance by Sector (2024)</b>', template='plotly_white', height=500, width=900,
    xaxis=dict(zeroline=True, zerolinewidth=2, zerolinecolor='black'))
fig2.show()
print(f"💾 Chart saved: charts/02_sector_balance_2024.png")

            Industry_Type  Value in 2024_import  Value in 2024_export  \
2  Others / Raw Materials               5947692               1810042   
1  Intermediate Materials                955821                 13691   
0   High-Tech & Machinery                897381                512708   
3      Textiles & Apparel                 98688                  7656   

   Trade_Balance  
2       -4137650  
1        -942130  
0        -384673  
3         -91032  


In [ ]:
# ============================================================
# STEP 4: THE "5-0 SCORE" - WHERE TURKEY DOMINATES
# ============================================================

# Identify the 5 biggest deficit sectors for Morocco
top5_deficit = sector_2024.head(5).copy()

print("🏆 THE '5-0' SCORE - Turkey's Top 5 Dominance Areas")
print("="*70)
for i, row in top5_deficit.iterrows():
    print(f"  ⚽ {row['Macro_Sector']}: Deficit = ${abs(row['Balance_2024']):,.0f}K | Coverage = {row['Coverage_2024_%']:.1f}%")

print(f"\n📌 TOTAL DEFICIT in Top 5 sectors: ${abs(top5_deficit['Balance_2024'].sum()):,.0f}K")

# --- Chart 3: The "5-0 Score" Visualization ---
fig3 = go.Figure()

fig3.add_trace(go.Bar(
    name='Morocco Imports from Turkey',
    x=top5_deficit['Macro_Sector'],
    y=top5_deficit['Import_2024'],
    marker_color='#e74c3c',
    text=[f'${v:,.0f}K' for v in top5_deficit['Import_2024']],
    textposition='auto'
))
fig3.add_trace(go.Bar(
    name='Morocco Exports to Turkey',
    x=top5_deficit['Macro_Sector'],
    y=top5_deficit['Export_2024'],
    marker_color='#27ae60',
    text=[f'${v:,.0f}K' for v in top5_deficit['Export_2024']],
    textposition='auto'
))

fig3.update_layout(
    title='<b>The "5-0 Score": Where Turkey Dominates Morocco (2024)</b><br><sub>Minister: "Les Turcs ta7ninkom 5-0"</sub>',
    barmode='group', template='plotly_white',
    yaxis_title='Trade Volume (USD Thousand)',
    legend=dict(orientation="h", yanchor="bottom", y=-0.15),
    height=500, width=900
)
fig3.write_image('charts/03_five_zero_score.png', scale=2)
fig3.show()

print(f"💾 Chart saved: charts/03_five_zero_score.png")

In [ ]:
# ============================================================
# STEP 5: TRADE DEFICIT EVOLUTION BY SECTOR (2021-2024)
# ============================================================

# Trend data
sector_trend = df.groupby('Macro_Sector')[[f'Balance_{y}' for y in years]].sum().reset_index()
sector_melted = pd.melt(sector_trend, id_vars=['Macro_Sector'],
                        value_vars=[f'Balance_{y}' for y in years],
                        var_name='Year', value_name='Balance')
sector_melted['Year'] = sector_melted['Year'].str.replace('Balance_', '')

# --- Chart 4: Line Chart - Deficit Evolution ---
fig4 = px.line(sector_melted, x='Year', y='Balance', color='Macro_Sector',
               markers=True, title='<b>Trade Deficit Evolution by Sector (2021-2024)</b>',
               labels={'Balance': 'Trade Balance (USD K)', 'Macro_Sector': 'Sector'})
fig4.update_layout(template='plotly_white', height=500, width=900,
                   legend=dict(orientation="h", yanchor="bottom", y=-0.3))
fig4.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)
fig4.write_image('charts/04_deficit_evolution.png', scale=2)
fig4.show()

# --- Chart 5: Heatmap - Trade Balance by Sector and Year ---
pivot_data = sector_melted.pivot(index='Macro_Sector', columns='Year', values='Balance')
plt.figure(figsize=(12, 6))
sns.heatmap(pivot_data, annot=True, fmt=',.0f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'label': 'Trade Balance (USD K)'})
plt.title('Heatmap: Trade Balance by Sector & Year (Morocco vs Turkey)', fontsize=14, fontweight='bold')
plt.ylabel('Sector')
plt.xlabel('Year')
plt.tight_layout()
plt.savefig('charts/05_heatmap_balance.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"💾 Charts saved: charts/04_deficit_evolution.png, charts/05_heatmap_balance.png")

Data Cleaning Completed successfully!
  Code              Sector  Import_2024  Export_2024
0  '01  Agriculture & Food            0            0
1  '02  Agriculture & Food            0           13
2  '03  Agriculture & Food          267          429
3  '04  Agriculture & Food         1270            3
4  '05      Other Products          246        22470


In [ ]:
# ============================================================
# STEP 6: TOP 10 PRODUCTS - BIGGEST GAPS
# ============================================================

# Top 10 deficit products (where Turkey dominates Morocco the most)
top10_deficit = df.nlargest(10, 'Import_2024')[['Code_Clean', 'Product_Label', 'Import_2024', 'Export_2024', 'Balance_2024', 'Macro_Sector']].copy()
top10_deficit['Deficit'] = abs(top10_deficit['Balance_2024'])

print("🔴 TOP 10 PRODUCTS: Turkey's Biggest Penetration into Morocco (2024)")
print("="*90)
for i, (_, row) in enumerate(top10_deficit.iterrows(), 1):
    label = str(row['Product_Label'])[:50] if pd.notna(row['Product_Label']) else 'N/A'
    print(f"  {i}. HS {row['Code_Clean']} | {label}")
    print(f"     Import: ${row['Import_2024']:,.0f}K | Export: ${row['Export_2024']:,.0f}K | Gap: ${row['Deficit']:,.0f}K")

# Top 10 surplus products (where Morocco fights back)
top10_surplus = df.nlargest(10, 'Balance_2024')[['Code_Clean', 'Product_Label', 'Import_2024', 'Export_2024', 'Balance_2024', 'Macro_Sector']].copy()

print(f"\n🟢 TOP 10 PRODUCTS: Where Morocco Has Competitive Advantage")
print("="*90)
for i, (_, row) in enumerate(top10_surplus.iterrows(), 1):
    label = str(row['Product_Label'])[:50] if pd.notna(row['Product_Label']) else 'N/A'
    print(f"  {i}. HS {row['Code_Clean']} | {label}")
    print(f"     Export: ${row['Export_2024']:,.0f}K | Import: ${row['Import_2024']:,.0f}K | Surplus: ${row['Balance_2024']:,.0f}K")

# --- Chart 6: Top 10 Deficit Products ---
fig6 = go.Figure()
fig6.add_trace(go.Bar(y=top10_deficit['Product_Label'].str[:40], x=top10_deficit['Import_2024'],
                       name='Morocco Imports', orientation='h', marker_color='#e74c3c'))
fig6.add_trace(go.Bar(y=top10_deficit['Product_Label'].str[:40], x=top10_deficit['Export_2024'],
                       name='Morocco Exports', orientation='h', marker_color='#2ecc71'))
fig6.update_layout(
    title="<b>Top 10 Products: Turkey's Dominance Over Morocco (2024)</b>",
    barmode='group', template='plotly_white', height=600, width=900,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15),
    xaxis_title='Value (USD Thousand)'
)
fig6.write_image('charts/06_top10_deficit_products.png', scale=2)
fig6.show()

# --- Chart 7: Top 10 Surplus Products (Morocco's Strengths) ---
fig7 = go.Figure()
fig7.add_trace(go.Bar(y=top10_surplus['Product_Label'].str[:40], x=top10_surplus['Export_2024'],
                       name='Morocco Exports', orientation='h', marker_color='#2ecc71'))
fig7.add_trace(go.Bar(y=top10_surplus['Product_Label'].str[:40], x=top10_surplus['Import_2024'],
                       name='Morocco Imports', orientation='h', marker_color='#e74c3c'))
fig7.update_layout(
    title="<b>Top 10 Products: Morocco's Competitive Advantage vs Turkey (2024)</b>",
    barmode='group', template='plotly_white', height=600, width=900,
    legend=dict(orientation="h", yanchor="bottom", y=-0.15),
    xaxis_title='Value (USD Thousand)'
)
fig7.write_image('charts/07_top10_surplus_products.png', scale=2)
fig7.show()

print(f"\n💾 Charts saved: charts/06_top10_deficit_products.png, charts/07_top10_surplus_products.png")

--- 2024 Analysis Summary ---
                     Sector  Trade_Balance_2024  Coverage_Ratio_%
4            Other Products            -4559643             26.58
3       Metals (Iron/Steel)             -622516              0.78
2      Machinery & Vehicles             -384673             57.13
5       Textiles & Clothing             -108189             10.87
0        Agriculture & Food              -32792             11.20
1  Fertilizers (Phosphates)              152328          2,627.41


In [ ]:
# ============================================================
# STEP 7: COVERAGE RATIO ANALYSIS & RADAR CHART
# ============================================================

# Coverage ratio by macro sector
coverage = sector_2024[['Macro_Sector', 'Coverage_2024_%']].sort_values('Coverage_2024_%', ascending=False)

print("📊 COVERAGE RATIO BY SECTOR (Exports/Imports × 100)")
print("="*60)
print("   100% = balanced | <100% = deficit | >100% = surplus")
print("-"*60)
for _, row in coverage.iterrows():
    status = "✅ SURPLUS" if row['Coverage_2024_%'] > 100 else "🔴 DEFICIT"
    print(f"  {row['Macro_Sector']:30s} {row['Coverage_2024_%']:8.1f}%  {status}")

# --- Chart 8: Radar Chart - Sector Competitiveness ---
categories = coverage['Macro_Sector'].tolist()
values = coverage['Coverage_2024_%'].tolist()

fig8 = go.Figure()
fig8.add_trace(go.Scatterpolar(
    r=values + [values[0]],
    theta=categories + [categories[0]],
    fill='toself', name='Morocco Coverage Ratio (%)',
    fillcolor='rgba(231, 76, 60, 0.3)',
    line=dict(color='#e74c3c', width=2)
))
fig8.add_trace(go.Scatterpolar(
    r=[100]*len(categories) + [100],
    theta=categories + [categories[0]],
    name='Balance Line (100%)',
    line=dict(color='#2c3e50', width=1, dash='dash')
))
fig8.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, max(values)*1.2])),
    title='<b>Radar: Morocco\'s Export Coverage vs Turkey by Sector (2024)</b><br><sub>Inside dotted circle = Deficit | Outside = Surplus</sub>',
    template='plotly_white', height=550, width=700
)
fig8.write_image('charts/08_radar_coverage.png', scale=2)
fig8.show()

print(f"\n💾 Chart saved: charts/08_radar_coverage.png")

In [ ]:
# ============================================================
# STEP 8: GROWTH DYNAMICS - WHO IS GAINING MOMENTUM?
# ============================================================

# Year-over-year growth
growth_data = []
for y1, y2 in zip(years[:-1], years[1:]):
    imp_growth = (df[f'Import_{y2}'].sum() - df[f'Import_{y1}'].sum()) / (df[f'Import_{y1}'].sum() + 1) * 100
    exp_growth = (df[f'Export_{y2}'].sum() - df[f'Export_{y1}'].sum()) / (df[f'Export_{y1}'].sum() + 1) * 100
    growth_data.append({'Period': f'{y1}-{y2}', 'Import_Growth_%': imp_growth, 'Export_Growth_%': exp_growth})

growth_df = pd.DataFrame(growth_data)
growth_df['Gap_Widening'] = growth_df['Import_Growth_%'] - growth_df['Export_Growth_%']

print("📈 YEAR-OVER-YEAR GROWTH RATES")
print("="*70)
print(growth_df.to_string(index=False))

# --- Chart 9: Growth Rate Comparison ---
fig9 = go.Figure()
fig9.add_trace(go.Bar(x=growth_df['Period'], y=growth_df['Import_Growth_%'],
                       name="Turkey's Export Growth to Morocco", marker_color='#e74c3c'))
fig9.add_trace(go.Bar(x=growth_df['Period'], y=growth_df['Export_Growth_%'],
                       name="Morocco's Export Growth to Turkey", marker_color='#2ecc71'))
fig9.update_layout(
    title='<b>Year-over-Year Growth: Who Is Gaining Momentum?</b>',
    yaxis_title='Growth Rate (%)', barmode='group',
    template='plotly_white', height=450, width=800
)
fig9.write_image('charts/09_growth_comparison.png', scale=2)
fig9.show()

# --- Chart 10: Treemap of Import Composition ---
treemap_data = df[df['Import_2024'] > 0][['Macro_Sector', 'Sector', 'Import_2024']].copy()
treemap_data = treemap_data[treemap_data['Import_2024'] > 1000]  # Filter small values

fig10 = px.treemap(treemap_data, path=['Macro_Sector', 'Sector'], values='Import_2024',
                    color='Import_2024', color_continuous_scale='Reds',
                    title="<b>What Morocco Buys from Turkey (2024) - Treemap</b>")
fig10.update_layout(height=550, width=900)
fig10.write_image('charts/10_treemap_imports.png', scale=2)
fig10.show()

# --- Chart 11: Treemap of Export Composition ---
treemap_exp = df[df['Export_2024'] > 0][['Macro_Sector', 'Sector', 'Export_2024']].copy()
treemap_exp = treemap_exp[treemap_exp['Export_2024'] > 1000]

fig11 = px.treemap(treemap_exp, path=['Macro_Sector', 'Sector'], values='Export_2024',
                    color='Export_2024', color_continuous_scale='Greens',
                    title="<b>What Morocco Sells to Turkey (2024) - Treemap</b>")
fig11.update_layout(height=550, width=900)
fig11.write_image('charts/11_treemap_exports.png', scale=2)
fig11.show()

print(f"\n💾 Charts saved: 09, 10, 11")

In [ ]:
# ============================================================
# STEP 9: STRATEGIC DEPENDENCY INDEX
# ============================================================

# How dependent is Morocco on Turkish imports in each sector?
dependency = df.groupby('Macro_Sector').agg({
    'Import_2024': 'sum', 'Export_2024': 'sum'
}).reset_index()
dependency['Dependency_Index'] = dependency['Import_2024'] / (dependency['Import_2024'] + dependency['Export_2024'] + 1) * 100
dependency['Net_Position'] = np.where(dependency['Export_2024'] > dependency['Import_2024'], 'Exporter', 'Importer')
dependency = dependency.sort_values('Dependency_Index', ascending=False)

print("🔐 STRATEGIC DEPENDENCY INDEX")
print("="*70)
print("   Index > 50% = Morocco is a net importer (dependent)")
print("   Index < 50% = Morocco is a net exporter (dominant)")
print("-"*70)
for _, row in dependency.iterrows():
    bar_len = int(row['Dependency_Index'] / 2)
    bar = '█' * bar_len + '░' * (50 - bar_len)
    print(f"  {row['Macro_Sector']:25s} [{bar}] {row['Dependency_Index']:.1f}% ({row['Net_Position']})")

# --- Chart 12: Dependency Gauge ---
fig12 = go.Figure()
for i, (_, row) in enumerate(dependency.iterrows()):
    color = '#e74c3c' if row['Dependency_Index'] > 70 else '#f39c12' if row['Dependency_Index'] > 50 else '#2ecc71'
    fig12.add_trace(go.Bar(
        y=[row['Macro_Sector']], x=[row['Dependency_Index']],
        orientation='h', marker_color=color, showlegend=False,
        text=f"{row['Dependency_Index']:.1f}%", textposition='outside'
    ))
fig12.add_vline(x=50, line_dash="dash", line_color="black", line_width=2,
                annotation_text="Balance Line", annotation_position="top")
fig12.update_layout(
    title="<b>Strategic Dependency Index: Morocco's Reliance on Turkish Imports</b><br><sub>Red >70% = High dependency | Yellow >50% = Moderate | Green <50% = Morocco dominates</sub>",
    xaxis_title="Dependency Index (%)", xaxis_range=[0, 105],
    template='plotly_white', height=450, width=900
)
fig12.write_image('charts/12_dependency_index.png', scale=2)
fig12.show()

print(f"\n💾 Chart saved: charts/12_dependency_index.png")

In [ ]:
# ============================================================
# STEP 10: HIGH-TECH GAP DEEP DIVE
# ============================================================

# The minister specifically mentioned engineering & industrial sectors
hightech = df[df['Macro_Sector'] == 'High-Tech & Machinery'].copy()
hightech = hightech.sort_values('Balance_2024')

print("🔧 HIGH-TECH & MACHINERY - DEEP DIVE (The Core of the '5-0')")
print("="*80)
total_ht_import = hightech['Import_2024'].sum()
total_ht_export = hightech['Export_2024'].sum()
total_ht_gap = total_ht_export - total_ht_import

print(f"  Total High-Tech Imports from Turkey: ${total_ht_import:,.0f}K")
print(f"  Total High-Tech Exports to Turkey:   ${total_ht_export:,.0f}K")
print(f"  HIGH-TECH GAP:                       ${total_ht_gap:,.0f}K")
print(f"  Coverage Ratio:                      {(total_ht_export/(total_ht_import+1))*100:.1f}%")
print(f"\n  ⚠️  This is where the '5-0' is most visible!")
print(f"\n  Detailed breakdown:")
for _, row in hightech.iterrows():
    label = str(row['Product_Label'])[:45] if pd.notna(row['Product_Label']) else 'N/A'
    print(f"    HS {row['Code_Clean']:4s} | {label:47s} | Gap: ${row['Balance_2024']:>12,.0f}K")

# --- Chart 13: High-Tech Gap Breakdown ---
fig13 = go.Figure()
fig13.add_trace(go.Bar(
    y=hightech['Sector'], x=hightech['Import_2024'],
    name='Morocco Imports', orientation='h', marker_color='#e74c3c'
))
fig13.add_trace(go.Bar(
    y=hightech['Sector'], x=hightech['Export_2024'],
    name='Morocco Exports', orientation='h', marker_color='#2ecc71'
))
fig13.update_layout(
    title="<b>High-Tech & Machinery: The Heart of the '5-0' Score</b><br><sub>Turkey dominates in machinery, electronics, vehicles & precision instruments</sub>",
    barmode='group', template='plotly_white', height=400, width=900,
    xaxis_title='Value (USD Thousand)',
    legend=dict(orientation="h", yanchor="bottom", y=-0.15)
)
fig13.write_image('charts/13_hightech_gap.png', scale=2)
fig13.show()

# --- Chart 14: Pie chart of Morocco's import composition ---
imp_composition = df.groupby('Macro_Sector')['Import_2024'].sum().reset_index()
imp_composition = imp_composition[imp_composition['Import_2024'] > 0]

fig14 = px.pie(imp_composition, values='Import_2024', names='Macro_Sector',
               title="<b>Composition of Morocco's Imports from Turkey (2024)</b>",
               color_discrete_sequence=px.colors.qualitative.Set2)
fig14.update_traces(textposition='inside', textinfo='percent+label')
fig14.update_layout(height=500, width=700)
fig14.write_image('charts/14_import_composition_pie.png', scale=2)
fig14.show()

print(f"\n💾 Charts saved: charts/13_hightech_gap.png, charts/14_import_composition_pie.png")

In [ ]:
# ============================================================
# STEP 11: MOROCCO'S STRENGTHS - WHERE WE FIGHT BACK
# ============================================================

# Sectors where Morocco has a SURPLUS
surplus_sectors = sector_2024[sector_2024['Balance_2024'] > 0].sort_values('Balance_2024', ascending=False)
deficit_sectors = sector_2024[sector_2024['Balance_2024'] < 0].sort_values('Balance_2024')

print("🟢 MOROCCO'S COMPETITIVE ADVANTAGES vs TURKEY")
print("="*70)
if len(surplus_sectors) > 0:
    for _, row in surplus_sectors.iterrows():
        print(f"  ✅ {row['Macro_Sector']}: Surplus of ${row['Balance_2024']:,.0f}K (Coverage: {row['Coverage_2024_%']:.0f}%)")
else:
    print("  ⚠️  Morocco has NO sector with trade surplus vs Turkey!")

print(f"\n🔴 SECTORS REQUIRING URGENT ACTION")
print("-"*70)
for _, row in deficit_sectors.iterrows():
    print(f"  ❌ {row['Macro_Sector']}: Deficit of ${abs(row['Balance_2024']):,.0f}K (Coverage: {row['Coverage_2024_%']:.1f}%)")

# --- Chart 15: Surplus vs Deficit Donut ---
labels = ['Total Deficit Sectors', 'Total Surplus Sectors']
values = [abs(deficit_sectors['Balance_2024'].sum()), max(surplus_sectors['Balance_2024'].sum(), 0) if len(surplus_sectors) > 0 else 0]

fig15 = go.Figure(data=[go.Pie(
    labels=labels, values=values, hole=0.5,
    marker_colors=['#e74c3c', '#2ecc71'],
    textinfo='label+value+percent'
)])
fig15.update_layout(
    title="<b>Morocco's Trade Position: Deficit vs Surplus Magnitude</b>",
    template='plotly_white', height=450, width=700,
    annotations=[dict(text=f'Net: ${(values[1]-values[0]):,.0f}K', x=0.5, y=0.5, font_size=14, showarrow=False)]
)
fig15.write_image('charts/15_deficit_vs_surplus.png', scale=2)
fig15.show()

print(f"\n💾 Chart saved: charts/15_deficit_vs_surplus.png")

In [ ]:
# ============================================================
# STEP 12: FINAL SCOREBOARD & KEY FINDINGS
# ============================================================

total_imp = df['Import_2024'].sum()
total_exp = df['Export_2024'].sum()
total_bal = total_exp - total_imp
coverage = (total_exp / total_imp) * 100

# Count sectors
n_surplus = len(sector_2024[sector_2024['Balance_2024'] > 0])
n_deficit = len(sector_2024[sector_2024['Balance_2024'] < 0])

print("╔" + "═"*68 + "╗")
print("║" + "  FINAL SCOREBOARD: MOROCCO vs TURKEY (2024)".center(68) + "║")
print("╠" + "═"*68 + "╣")
print(f"║  Total Imports from Turkey:    ${total_imp:>15,.0f}K".ljust(69) + "║")
print(f"║  Total Exports to Turkey:      ${total_exp:>15,.0f}K".ljust(69) + "║")
print(f"║  NET TRADE BALANCE:            ${total_bal:>15,.0f}K".ljust(69) + "║")
print(f"║  Coverage Ratio:               {coverage:>14.1f}%".ljust(69) + "║")
print(f"║  Sectors in Deficit:           {n_deficit:>15d}".ljust(69) + "║")
print(f"║  Sectors in Surplus:           {n_surplus:>15d}".ljust(69) + "║")
print("╠" + "═"*68 + "╣")
print("║" + "  KEY FINDINGS".center(68) + "║")
print("╠" + "═"*68 + "╣")

findings = [
    f"1. Morocco imports {total_imp/total_exp:.1f}x more than it exports to Turkey",
    f"2. The trade deficit reached ${abs(total_bal):,.0f}K in 2024",
    f"3. High-Tech & Machinery is the biggest deficit sector",
    f"4. Turkey dominates in {n_deficit} out of {n_deficit+n_surplus} sectors",
    f"5. Morocco's main strength: Fertilizers (Phosphates)",
    f"6. The '5-0 Score' reflects structural industrial weakness",
    f"7. Textiles sector shows Turkey's manufacturing dominance",
    f"8. Energy imports add significant weight to the deficit",
]
for f in findings:
    print(f"║  {f}".ljust(69) + "║")

print("╠" + "═"*68 + "╣")
print("║" + "  RECOMMENDATIONS".center(68) + "║")
print("╠" + "═"*68 + "╣")

recommendations = [
    "→ Invest in high-tech manufacturing & local production",
    "→ Leverage phosphates advantage for strategic trade deals",
    "→ Develop automotive & aerospace supply chains",
    "→ Reduce dependency on Turkish textiles via local industry",
    "→ Target African markets (3B people) as the minister urged",
    "→ Foster engineering talent toward industrial innovation",
]
for r in recommendations:
    print(f"║  {r}".ljust(69) + "║")

print("╚" + "═"*68 + "╝")

# --- Chart 16: Final Summary Dashboard ---
fig16 = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Trade Balance Evolution', 'Sector Breakdown 2024',
                    'Import vs Export 2024', 'Coverage Ratio by Sector'),
    specs=[[{"type": "scatter"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}]]
)

# Panel 1: Balance evolution
for y in years:
    bal = df[f'Balance_{y}'].sum()
summary_vals = [df[f'Balance_{y}'].sum() for y in years]
fig16.add_trace(go.Scatter(x=years, y=summary_vals, mode='lines+markers',
                            name='Trade Balance', line=dict(color='#e74c3c', width=3)), row=1, col=1)
fig16.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=1)

# Panel 2: Sector balance
fig16.add_trace(go.Bar(x=sector_2024['Macro_Sector'], y=sector_2024['Balance_2024'],
                        marker_color=['#e74c3c' if v < 0 else '#2ecc71' for v in sector_2024['Balance_2024']],
                        name='Balance 2024', showlegend=False), row=1, col=2)

# Panel 3: Import vs Export
fig16.add_trace(go.Bar(x=['Imports', 'Exports'], y=[total_imp, total_exp],
                        marker_color=['#e74c3c', '#2ecc71'], showlegend=False,
                        text=[f'${total_imp:,.0f}K', f'${total_exp:,.0f}K'],
                        textposition='auto'), row=2, col=1)

# Panel 4: Coverage ratio
cov_data = sector_2024.sort_values('Coverage_2024_%')
fig16.add_trace(go.Bar(x=cov_data['Macro_Sector'], y=cov_data['Coverage_2024_%'],
                        marker_color=['#e74c3c' if v < 100 else '#2ecc71' for v in cov_data['Coverage_2024_%']],
                        showlegend=False), row=2, col=2)
fig16.add_hline(y=100, line_dash="dash", line_color="black", row=2, col=2)

fig16.update_layout(
    title='<b>MOROCCO vs TURKEY: Executive Dashboard (2024)</b>',
    template='plotly_white', height=700, width=1100,
    showlegend=False
)
fig16.write_image('charts/16_executive_dashboard.png', scale=2)
fig16.show()

# --- Save data to Excel for the report ---
with pd.ExcelWriter('report/trade_analysis_data.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Full_Data', index=False)
    sector_2024.to_excel(writer, sheet_name='Sector_Analysis', index=False)
    top10_deficit.to_excel(writer, sheet_name='Top10_Deficit', index=False)
    top10_surplus.to_excel(writer, sheet_name='Top10_Surplus', index=False)
    dependency.to_excel(writer, sheet_name='Dependency_Index', index=False)

print(f"\n💾 Dashboard saved: charts/16_executive_dashboard.png")
print(f"📊 Excel data saved: report/trade_analysis_data.xlsx")
print(f"\n✅ ANALYSIS COMPLETE - {len(os.listdir('charts'))} charts generated!")